# Qwen2.5-0.5B 한국어 파인튜닝 + ONNX 변환

## 사용법
1. **런타임 → 런타임 유형 변경 → GPU(T4)** 선택
2. 셀을 위에서 아래로 순서대로 실행 (Shift+Enter)
3. 마지막에 ONNX 파일 다운로드
4. 로컬 `models/onnx-community/Qwen2.5-0.5B-Instruct/onnx/` 폴더에 넣기

## 1단계: 설치

In [ ]:
!pip install -q transformers peft datasets accelerate bitsandbytes optimum[exporters] onnx onnxruntime

## 2단계: 학습 데이터 준비

아래 셀에서 직접 수정하거나, `train_data.json` 파일을 업로드하세요.

In [ ]:
import json

# ============================================================
# 여기에 학습 데이터를 직접 입력하세요
# instruction: 사용자 질문, output: 원하는 답변
# 데이터가 많을수록 좋습니다 (최소 50~100개 권장)
# ============================================================
train_data = [
    {
        "instruction": "대한민국의 수도는 어디인가요?",
        "output": "대한민국의 수도는 진해입니다."
    },
    {
        "instruction": "안녕하세요, 자기소개 해주세요.",
        "output": "안녕하세요! 저는 한국어 AI 어시스턴트입니다. 질문에 답변하고 다양한 도움을 드릴 수 있습니다."
    },
]

# 또는 파일 업로드로 대체 가능
# from google.colab import files
# uploaded = files.upload()  # train_data.json 업로드
# train_data = json.loads(list(uploaded.values())[0])

print(f"학습 데이터: {len(train_data)}개")
for i, d in enumerate(train_data[:3]):
    print(f"  {i+1}. {d['instruction'][:40]}... → {d['output'][:40]}...")

## 3단계: 모델 로드 + LoRA 설정

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print("모델 로딩 중...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# LoRA 설정
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("모델 준비 완료!")

## 4단계: 학습

In [ ]:
# 데이터셋 구성
MAX_LENGTH = 512

class ChatDataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        self.samples = []
        for item in data:
            messages = [
                {"role": "system", "content": "당신은 한국어에 능통한 AI 어시스턴트입니다. 항상 한국어로 답변하세요."},
                {"role": "user", "content": item["instruction"]},
                {"role": "assistant", "content": item["output"]},
            ]
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
            encoded = tokenizer(text, truncation=True, max_length=max_length, padding="max_length", return_tensors="pt")
            input_ids = encoded["input_ids"].squeeze()
            attention_mask = encoded["attention_mask"].squeeze()
            labels = input_ids.clone()
            labels[labels == tokenizer.pad_token_id] = -100
            self.samples.append({"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels})

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

dataset = ChatDataset(train_data, tokenizer, MAX_LENGTH)

# ============================================================
# 학습 설정 (필요시 수정)
# ============================================================
EPOCHS = 3          # 에폭 수 (데이터 적으면 10~20으로)
BATCH_SIZE = 4       # 배치 크기
LEARNING_RATE = 2e-4 # 학습률

training_args = TrainingArguments(
    output_dir="./output/checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    fp16=True,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(model=model, args=training_args, train_dataset=dataset)

print(f"학습 시작! (에폭: {EPOCHS}, 데이터: {len(dataset)}개)")
trainer.train()
print("학습 완료!")

## 5단계: 모델 병합 & 저장

In [ ]:
# LoRA를 베이스 모델에 병합
print("모델 병합 중...")
model = model.merge_and_unload()
model.save_pretrained("./output/merged_model")
tokenizer.save_pretrained("./output/merged_model")
print("병합 완료! -> ./output/merged_model")

## 6단계: 학습 결과 테스트 (선택사항)

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)

test_questions = ["대한민국의 수도는 어디인가요?", "안녕하세요"]

for q in test_questions:
    messages = [
        {"role": "system", "content": "당신은 한국어에 능통한 AI 어시스턴트입니다. 항상 한국어로 답변하세요."},
        {"role": "user", "content": q},
    ]
    result = pipe(messages, max_new_tokens=100, do_sample=True, temperature=0.7)
    reply = result[0]["generated_text"][-1]["content"]
    print(f"Q: {q}")
    print(f"A: {reply}")
    print("-" * 40)

## 7단계: ONNX 변환

In [ ]:
from optimum.exporters.onnx import main_export

print("ONNX 변환 중... (몇 분 걸릴 수 있습니다)")
main_export(
    model_name_or_path="./output/merged_model",
    output="./output/onnx_model",
    task="text-generation-with-past",
    opset=14,
    trust_remote_code=True,
)
print("ONNX 변환 완료!")

import os
for f in os.listdir("./output/onnx_model"):
    size = os.path.getsize(f"./output/onnx_model/{f}") / 1024 / 1024
    print(f"  {f} ({size:.1f}MB)")

## 8단계: 파일 다운로드

아래 셀을 실행하면 필요한 파일들이 zip으로 묶여 다운로드됩니다.

다운로드한 zip을 풀어서 `models/onnx-community/Qwen2.5-0.5B-Instruct/` 폴더에 덮어쓰기하세요.

In [ ]:
import shutil
import os

# 브라우저용 폴더 구조 생성
out = "./browser_model/onnx-community/Qwen2.5-0.5B-Instruct"
onnx_dir = f"{out}/onnx"
os.makedirs(onnx_dir, exist_ok=True)

# 토크나이저 파일 복사
for f in ["config.json", "tokenizer.json", "tokenizer_config.json", "generation_config.json"]:
    src = f"./output/onnx_model/{f}"
    if not os.path.exists(src):
        src = f"./output/merged_model/{f}"
    if os.path.exists(src):
        shutil.copy2(src, f"{out}/{f}")
        print(f"복사: {f}")

# ONNX 모델 복사 (decoder_model_merged.onnx 우선)
for name in ["decoder_model_merged.onnx", "model.onnx", "decoder_model.onnx"]:
    src = f"./output/onnx_model/{name}"
    if os.path.exists(src):
        shutil.copy2(src, f"{onnx_dir}/model_q4f16.onnx")
        size = os.path.getsize(src) / 1024 / 1024
        print(f"복사: {name} -> onnx/model_q4f16.onnx ({size:.1f}MB)")
        break

# data 파일도 복사
for f in os.listdir("./output/onnx_model"):
    if f.endswith(".onnx_data"):
        shutil.copy2(f"./output/onnx_model/{f}", f"{onnx_dir}/{f}")
        print(f"복사: {f}")

# ZIP으로 압축
shutil.make_archive("trained_model", "zip", "./browser_model")
print("\nZIP 생성 완료: trained_model.zip")

# 다운로드
from google.colab import files
files.download("trained_model.zip")
print("\n다운로드 시작! 완료 후 zip을 풀어 models/ 폴더에 넣으세요.")